In [96]:
""" 
Import all the libraries required for transaction reconciliation pipelines.
"""

import pandas as pd
import numpy as np
import re

from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

In [97]:
"""
Load the bank and register datasets.

These contains transaction records from two sources.
We will use them later to find the matching transactions.
"""

bank = pd.read_csv("../data/bank_statements.csv")
register = pd.read_csv("../data/check_register.csv")

print("Bank dataset shape:", bank.shape)
print("Register dataset shape:", register.shape)

bank.head()

Bank dataset shape: (308, 6)
Register dataset shape: (308, 7)


,transaction_id,date,description,amount,type,balance
0,B0047,2023-01-01,BP GAS #1775,46.48,DEBIT,4953.52
1,B0092,2023-01-01,BP GAS #4099,51.53,DEBIT,4901.99
2,B0015,2023-01-02,CAFE #6311,108.21,DEBIT,4793.78
3,B0084,2023-01-02,TRADER JOES,137.35,DEBIT,4656.43
4,B0142,2023-01-04,KROGER #6864,198.14,DEBIT,4458.29


In [98]:
"""
Convert date columns to datetime so we cancompare dates later.
"""
bank["date"] = pd.to_datetime(bank["date"])
register["date"] = pd.to_datetime(register["date"])

print("Date columns converted")

Date columns converted


In [99]:
def clean_text(text):
    """
    Clean transaction descriptions before generating embeddings.
    """

    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]", "", text)
    return text.strip()


bank["description"] = bank["description"].apply(clean_text)
register["description"] = register["description"].apply(clean_text)

print("Example cleaned descriptions:")
print(bank["description"].head())


Example cleaned descriptions:
0     bpgas1775
1     bpgas4099
2      cafe6311
3    traderjoes
4    kroger6864
Name: description, dtype: str


In [100]:
"""
Standardize transaction type labels so both datasets use the same values.
"""

bank["type"] = bank["type"].replace({
    "DEBIT" : "DR",
    "CREDIT" : "CR"
})

print("Transaction types normalized.")

Transaction types normalized.


In [101]:
"""
Inspect dataset statistics.
"""

print("Bank rows:", len(bank))
print("Register rows:", len(register))

bank["amount"].describe()

Bank rows: 308
Register rows: 308


count     308.000000
mean      353.743799
std       693.453011
min        12.710000
25%        65.832500
50%       135.105000
75%       235.587500
max      4437.990000
Name: amount, dtype: float64

In [102]:
"""
Create pseudo ground truth pairs using unique transaction amounts.

If an amount appears exactly once in both datasets,
we assume those transactions correspond.

Since the dataset has no explicit ID column,
the dataframe index is used as the transaction identifier.
"""

bank_counts = bank["amount"].value_counts()
register_counts = register["amount"].value_counts()

test_pairs = []

for amt in bank_counts.index:

    if bank_counts[amt] == 1 and register_counts.get(amt, 0) == 1:

        bank_row = bank[bank["amount"] == amt].iloc[0]
        reg_row = register[register["amount"] == amt].iloc[0]

        test_pairs.append({
            "bank_id": bank_row.name,
            "register_id": reg_row.name
        })

test_pairs = pd.DataFrame(test_pairs)

print("Ground truth pairs:", len(test_pairs))
test_pairs.head()

Ground truth pairs: 286


,bank_id,register_id
0,0,2
1,1,3
2,2,1
3,3,0
4,4,4


In [103]:
"""
Remove ground truth transactions so the algorithm
predicts the remaining difficult matches.
"""

bank_remaining = bank.drop(index=test_pairs["bank_id"])
register_remaining = register.drop(index=test_pairs["register_id"])

print("Remaining bank transactions:", len(bank_remaining))
print("Remaining register transactions:", len(register_remaining))

Remaining bank transactions: 22
Remaining register transactions: 22


In [104]:
"""
Keep full datasets for prediction.

Ground truth pairs are only used for evaluation,
not removed from the dataset.
"""

bank_remaining = bank.copy()
register_remaining = register.copy()

print("Bank transactions:", len(bank_remaining))
print("Register transactions:", len(register_remaining))

Bank transactions: 308
Register transactions: 308


In [105]:
"""
Generate embeddings from transaction descriptions.
"""

model = SentenceTransformer("all-MiniLM-L6-v2")

bank_embeddings = model.encode(bank_remaining["description"].astype(str).tolist())
register_embeddings = model.encode(register_remaining["description"].astype(str).tolist())

print("Embeddings created.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5236.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings created.


In [106]:
"""
Find candidate matches using nearest neighbor search.
"""

nn = NearestNeighbors(n_neighbors=3, metric="cosine")

nn.fit(register_embeddings)

distances, indices = nn.kneighbors(bank_embeddings)

print("Nearest neighbors calculated.")

Nearest neighbors calculated.


In [107]:
"""
Match transactions using amount blocking and similarity scoring.
"""

matches = []

used_registers = set()

bank_rows = bank.reset_index()
register_rows = register.reset_index()

for i, bank_row in bank_rows.iterrows():

    best_score = -1
    best_match = None

    for j, reg_row in register_rows.iterrows():

        # block by amount
        if abs(bank_row["amount"] - reg_row["amount"]) > 2:
            continue

        # block by date
        date_diff = abs((bank_row["date"] - reg_row["date"]).days)
        if date_diff > 3:
            continue

        # prevent reuse of same register row
        if reg_row["index"] in used_registers:
            continue

        text_sim = cosine_similarity(
            [bank_embeddings[i]],
            [register_embeddings[j]]
        )[0][0]

        amount_score = 1 if abs(bank_row["amount"] - reg_row["amount"]) < 0.5 else 0
        date_score = 1 if date_diff <= 2 else 0
        type_score = 1 if bank_row["type"] == reg_row["type"] else 0

        score = (
            0.5 * text_sim +
            0.3 * amount_score +
            0.15 * date_score +
            0.05 * type_score
        )

        if score > best_score:
            best_score = score
            best_match = reg_row["index"]

    if best_match is not None:

        used_registers.add(best_match)

        matches.append({
            "bank_id": bank_row["index"],
            "register_id": best_match,
            "confidence": best_score
        })

matches = pd.DataFrame(matches)

print("Predicted matches:", len(matches))

Predicted matches: 283


In [108]:
"""
Evaluate predictions against ground truth pairs.
"""

correct = 0

true_pairs = {
    (row["bank_id"], row["register_id"])
    for _, row in test_pairs.iterrows()
}

pred_pairs = {
    (row["bank_id"], row["register_id"])
    for _, row in matches.iterrows()
}

for pair in pred_pairs:
    if pair in true_pairs:
        correct += 1

precision = correct / len(pred_pairs) if len(pred_pairs) > 0 else 0
recall = correct / len(true_pairs) if len(true_pairs) > 0 else 0

f1 = 0
if precision + recall > 0:
    f1 = 2 * precision * recall / (precision + recall)

print("Correct matches:", correct)
print("Total predicted:", len(pred_pairs))
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Correct matches: 264
Total predicted: 283
Precision: 0.9328621908127208
Recall: 0.9230769230769231
F1 Score: 0.9279437609841829


In [109]:
"""
Display example matches for manual inspection.
"""

for _, row in matches.head(20).iterrows():

    bank_row = bank.loc[row["bank_id"]]
    reg_row = register.loc[row["register_id"]]

    print("BANK:", bank_row["description"], bank_row["amount"], bank_row["date"])
    print("REG :", reg_row["description"], reg_row["amount"], reg_row["date"])
    print("Confidence:", round(row["confidence"], 3))
    print("-----")

BANK: bpgas1775 46.48 2023-01-01 00:00:00
REG : fillup 46.48 2022-12-31 00:00:00
Confidence: 0.6
-----
BANK: bpgas4099 51.53 2023-01-01 00:00:00
REG : fillup 51.53 2022-12-31 00:00:00
Confidence: 0.588
-----
BANK: cafe6311 108.21 2023-01-02 00:00:00
REG : dinnerout 108.21 2022-12-31 00:00:00
Confidence: 0.636
-----
BANK: traderjoes 137.35 2023-01-02 00:00:00
REG : groceries 137.35 2022-12-30 00:00:00
Confidence: 0.506
-----
BANK: kroger6864 198.14 2023-01-04 00:00:00
REG : foodshopping 198.14 2023-01-01 00:00:00
Confidence: 0.439
-----
BANK: kroger5496 158.43 2023-01-09 00:00:00
REG : grocerystore 158.43 2023-01-07 00:00:00
Confidence: 0.675
-----
BANK: bpgas7420 72.16 2023-01-10 00:00:00
REG : fuel 72.16 2023-01-10 00:00:00
Confidence: 0.66
-----
BANK: onlinepmtwater 147.29 2023-01-12 00:00:00
REG : electricbill 147.33 2023-01-10 00:00:00
Confidence: 0.636
-----
BANK: amazoncom 19.24 2023-01-14 00:00:00
REG : onlinepurchase 19.24 2023-01-12 00:00:00
Confidence: 0.694
-----
BANK: whole

In [110]:
"""
Inspect the lowest confidence matches.

These are the transactions a financial analyst would
review manually to verify correctness.
"""

low_conf = matches.sort_values("confidence").head(10)

print("Lowest confidence matches:")

for _, row in low_conf.iterrows():

    bank_row = bank.loc[row["bank_id"]]
    reg_row = register.loc[row["register_id"]]

    print("BANK:", bank_row["description"], bank_row["amount"], bank_row["date"])
    print("REG :", reg_row["description"], reg_row["amount"], reg_row["date"])
    print("Confidence:", round(row["confidence"], 3))
    print("-----")

Lowest confidence matches:
BANK: safeway2184 126.09 2023-04-06 00:00:00
REG : weeklygroceries 126.09 2023-04-03 00:00:00
Confidence: 0.383
-----
BANK: safeway8513 116.68 2023-04-13 00:00:00
REG : foodshopping 116.68 2023-04-10 00:00:00
Confidence: 0.396
-----
BANK: misctransaction9096 115.81 2023-03-09 00:00:00
REG : otherexpense 115.81 2023-03-06 00:00:00
Confidence: 0.428
-----
BANK: ck3062 346.88 2023-10-01 00:00:00
REG : paidbycheck 346.88 2023-09-28 00:00:00
Confidence: 0.439
-----
BANK: kroger6864 198.14 2023-01-04 00:00:00
REG : foodshopping 198.14 2023-01-01 00:00:00
Confidence: 0.439
-----
BANK: traderjoes 249.17 2023-03-23 00:00:00
REG : weeklygroceries 249.17 2023-03-20 00:00:00
Confidence: 0.443
-----
BANK: kroger6854 130.74 2023-09-19 00:00:00
REG : weeklygroceries 130.74 2023-09-16 00:00:00
Confidence: 0.451
-----
BANK: check2028 772.09 2023-05-28 00:00:00
REG : paidycheck 772.09 2023-05-25 00:00:00
Confidence: 0.454
-----
BANK: kroger6892 232.79 2023-08-29 00:00:00
REG :

In [111]:
"""
Analyze confidence score distribution.
"""

print(matches["confidence"].describe())

count    283.000000
mean       0.651266
std        0.091099
min        0.382789
25%        0.591230
50%        0.651084
75%        0.696796
max        1.000000
Name: confidence, dtype: float64


In [112]:
"""
Print final evaluation metrics.
"""

print("Final Results")
print("----------------")
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1 Score:", round(f1,4))

Final Results
----------------
Precision: 0.9329
Recall: 0.9231
F1 Score: 0.9279
